# Exercise 08 — MQTT: Topics, Wildcards & the Full Pipeline
### 🔗 Connecting everything together

This is the capstone exercise. You will:
1. Master MQTT topic hierarchies and wildcards (`+` and `#`)
2. Send **SenML**-formatted payloads over MQTT (combining ex04 + ex07)
3. Build a **bridge**: an MQTT subscriber that forwards data to the REST catalog from ex06

This is a real IoT architecture pattern: sensors speak MQTT (lightweight, fire-and-forget),
a gateway bridges into REST (for storage and querying).

```
 Sensor          Broker         Bridge          REST Catalog
  [TRN-001] --MQTT--> [HiveMQ] --MQTT--> [bridge.py] --HTTP PUT--> [localhost:5001]
```

---

In [ ]:
import paho.mqtt.client as mqtt
import json, time, requests, threading

BROKER     = "broker.hivemq.com"
PORT       = 1883
STUDENT_ID = "student_abc"    # <-- use the same value as in ex07!
BASE_TOPIC = f"smartcity/turin/{STUDENT_ID}"

print(f"Base topic: {BASE_TOPIC}")

## 1️⃣  Topic hierarchy — naming matters

Good topic design makes your system easy to filter and scale.
Your lecture shows: `/sensor/1234/temperature`, `/sensor/567/humidity`

Let's map our sensor network to a proper topic tree.

In [ ]:
# Topic tree for SmartCity Turin
# smartcity/turin/{student}/
#   {sensor_id}/
#     temperature
#     humidity
#     pm25
#     status          ← online/offline heartbeat

sensors = ["TRN-001", "TRN-002", "TRN-003", "TRN-004", "TRN-005"]
measurements = ["temperature", "humidity", "pm25"]

print("Full topic tree:")
for s in sensors:
    for m in measurements:
        print(f"  {BASE_TOPIC}/{s}/{m}")
    print(f"  {BASE_TOPIC}/{s}/status")
    print()

## 2️⃣  Wildcards: `+` (one level) and `#` (all remaining)

As your lecture explains:
- `+` matches exactly **one** level: `/sensor/+/temperature` → all sensors' temperature
- `#` matches **zero or more** levels: `/sensor/TRN-001/#` → everything from TRN-001

In [ ]:
# Let's verify wildcards work by creating selective subscribers

def make_subscriber(name, topic_filter):
    """Create a subscriber that prints what it receives."""
    log = []

    def on_connect(client, userdata, flags, rc):
        client.subscribe(topic_filter)

    def on_message(client, userdata, msg):
        entry = f"[{name}] {msg.topic} → {msg.payload.decode()}"
        print(f"  {entry}")
        log.append(entry)

    c = mqtt.Client(client_id=f"{STUDENT_ID}_{name}")
    c.on_connect = on_connect
    c.on_message = on_message
    c.connect(BROKER, PORT)
    c.loop_start()
    return c, log

# Three subscribers with different wildcard patterns
sub_all,   log_all   = make_subscriber("ALL   ", f"{BASE_TOPIC}/#")
sub_temp,  log_temp  = make_subscriber("TEMP  ", f"{BASE_TOPIC}/+/temperature")
sub_trn1,  log_trn1  = make_subscriber("TRN-001", f"{BASE_TOPIC}/TRN-001/#")

time.sleep(1)
print("Subscribers ready. Now publishing...\n")

In [ ]:
# Publisher
pub = mqtt.Client(client_id=f"{STUDENT_ID}_pub2")
pub.connect(BROKER, PORT)
pub.loop_start()
time.sleep(0.5)

readings = [
    ("TRN-001", "temperature", 22.4),
    ("TRN-001", "humidity",    58.0),
    ("TRN-001", "pm25",        12.1),
    ("TRN-002", "temperature", 23.1),
    ("TRN-002", "pm25",        18.4),
    ("TRN-003", "temperature", 21.0),
    ("TRN-003", "pm25",        35.2),
]

for sensor_id, meas, value in readings:
    topic = f"{BASE_TOPIC}/{sensor_id}/{meas}"
    pub.publish(topic, str(value), qos=1)
    time.sleep(0.15)

time.sleep(1.5)
print(f"\n--- Summary ---")
print(f"  '#' subscriber (all)         received: {len(log_all)}  messages")
print(f"  '+/temperature' subscriber   received: {len(log_temp)} messages")
print(f"  'TRN-001/#' subscriber       received: {len(log_trn1)} messages")
print()
print("As expected:")
print(f"  ALL={len(readings)}, TEMP=3 (one per sensor), TRN-001=3 (all its measurements)")

## 3️⃣  SenML payloads over MQTT

MQTT is protocol-agnostic — the payload can be anything.
Combining SenML (from ex04) with MQTT gives you a standardised, compact IoT message.

In [ ]:
def make_senml_payload(sensor_id, measurement, value, unit):
    """Build a SenML-formatted MQTT payload."""
    doc = {
        "bn": f"http://smartcity.turin.it/sensor/{sensor_id}/",
        "e":  [{"n": measurement, "u": unit, "t": time.time(), "v": value}]
    }
    return json.dumps(doc)

# Map measurements to SenML units
UNITS = {"temperature": "Cel", "humidity": "%RH", "pm25": "ug/m3"}

senml_received = []

def on_senml(client, userdata, msg):
    doc = json.loads(msg.payload.decode())
    bn  = doc["bn"]
    for event in doc["e"]:
        full_name = bn + event["n"]
        print(f"  📨 {full_name} = {event['v']} {event['u']}")
        senml_received.append(event)

sub_senml = mqtt.Client(client_id=f"{STUDENT_ID}_senml_sub")
sub_senml.on_message = on_senml
sub_senml.connect(BROKER, PORT)
sub_senml.subscribe(f"{BASE_TOPIC}/senml/#")
sub_senml.loop_start()
time.sleep(0.5)

print("Publishing SenML payloads...\n")
for sensor_id, meas, value in readings[:5]:
    topic   = f"{BASE_TOPIC}/senml/{sensor_id}/{meas}"
    payload = make_senml_payload(sensor_id, meas, value, UNITS[meas])
    pub.publish(topic, payload, qos=1)
    time.sleep(0.2)

time.sleep(1.5)
print(f"\nSenML events received: {len(senml_received)}")

## 4️⃣  The MQTT→REST Bridge

Now the capstone: an MQTT subscriber that forwards every sensor reading
to the REST catalog server from Exercise 06.

> **Make sure the server from Exercise 06 is still running (port 5001).**
> If not, re-run the server cell in that notebook first.

In [ ]:
REST_BASE = "http://localhost:5001"

# Check the REST server is alive
try:
    r = requests.get(f"{REST_BASE}/sensors", timeout=2)
    print(f"✅ REST Catalog reachable — {len(r.json())} sensors registered")
except Exception:
    print("❌ REST Catalog not running. Start the server in Exercise 06 first.")

In [ ]:
bridge_log = []

def bridge_on_message(client, userdata, msg):
    """
    MQTT → REST bridge.
    Topic format: {BASE_TOPIC}/{sensor_id}/{measurement}
    Payload: plain numeric value (string)
    """
    parts = msg.topic.split("/")
    if len(parts) < 2:
        return

    sensor_id   = parts[-2]   # e.g. TRN-001
    measurement = parts[-1]   # e.g. temperature

    try:
        value = float(msg.payload.decode())
    except ValueError:
        return   # skip non-numeric payloads

    # Map topic measurement names to REST field names
    field_map = {"temperature": "temperature_c", "humidity": "humidity_pct", "pm25": "pm25_ugm3"}
    field = field_map.get(measurement)
    if not field:
        return

    # Forward to REST catalog via PUT
    url  = f"{REST_BASE}/sensors/{sensor_id}"
    resp = requests.put(url, json={field: value}, timeout=2)

    entry = f"MQTT [{sensor_id}/{measurement}={value}] → REST PUT {resp.status_code}"
    print(f"  🔗 {entry}")
    bridge_log.append(entry)

bridge = mqtt.Client(client_id=f"{STUDENT_ID}_bridge")
bridge.on_message = bridge_on_message
bridge.connect(BROKER, PORT)
bridge.subscribe(f"{BASE_TOPIC}/+/+")   # all sensors, all measurements
bridge.loop_start()
time.sleep(0.5)
print("🔗 MQTT→REST bridge running\n")

# Now simulate sensors publishing — the bridge will forward to REST
live_readings = [
    ("TRN-001", "temperature", 23.9),
    ("TRN-001", "pm25",        14.0),
    ("TRN-002", "temperature", 24.1),
    ("TRN-003", "temperature", 20.5),
    ("TRN-003", "pm25",        38.0),   # high pollution!
    ("TRN-004", "humidity",    55.0),
]

for sensor_id, meas, value in live_readings:
    topic = f"{BASE_TOPIC}/{sensor_id}/{meas}"
    pub.publish(topic, str(value), qos=1)
    time.sleep(0.3)

time.sleep(2)
print(f"\n--- Bridge forwarded {len(bridge_log)} updates ---")

In [ ]:
# Verify the REST catalog reflects the MQTT updates
sensors = requests.get(f"{REST_BASE}/sensors").json()
print("Current state in REST catalog (after MQTT updates):\n")
print(f"  {'Sensor':<10} {'Location':<20} {'Temp (°C)':>10} {'PM2.5':>8}")
print("  " + "-" * 52)
for s in sensors:
    flag = " ⚠️" if s.get("pm25_ugm3", 0) > 20 else ""
    print(f"  {s['sensor_id']:<10} {s['location']:<20} {s.get('temperature_c', '-'):>10} {s.get('pm25_ugm3', '-'):>8}{flag}")

## 5️⃣  REST vs MQTT — when to use which?

Run this cell to see the conceptual summary.

In [ ]:
comparison = [
    ("Communication model", "Request/Response",   "Publish/Subscribe"),
    ("Who initiates",       "Client pulls data",  "Broker pushes data"),
    ("Coupling",            "Tight (client needs server address)", "Loose (via broker topic)"),
    ("Best for",            "Query, configure, store", "Real-time events, telemetry"),
    ("Delivery guarantee",  "HTTP status codes",  "QoS 0/1/2"),
    ("Payload format",      "JSON (structured)",  "Any (agnostic)"),
    ("IoT use case",        "Device catalog, dashboard", "Sensor streams, alerts"),
]

print(f"  {'Aspect':<25} {'REST/HTTP':<35} {'MQTT'}")
print("  " + "-" * 90)
for aspect, rest, mqtt_val in comparison:
    print(f"  {aspect:<25} {rest:<35} {mqtt_val}")

---
## 🏆 Challenges

1. Extend the bridge to also call `POST /sensors/<id>/history` (from Exercise 06) to log every reading in the history, not just update the current value.
2. Add a **Last Will and Testament (LWT)**: when a sensor disconnects unexpectedly, the broker should automatically publish `{"status": "offline"}` to `{BASE_TOPIC}/{sensor_id}/status`.
3. Create a `GET /alerts` REST endpoint (in ex06's server) that reads the current sensor list and returns all sensors with PM2.5 > 20. Then write an MQTT subscriber that calls this endpoint every 30 seconds and prints a report.
4. Describe in words: in the full SmartCity pipeline (sensors → MQTT → bridge → REST → client), what happens if the REST server goes down for 5 minutes? What MQTT feature could help recover?

In [ ]:
# Your code here
